In [ ]:
# ============================================
# JUPYTER NOTEBOOK - GraphRAG con Ollama + RDF
# ============================================

# ============================================
# CELDA 1 - IMPORTS
# ============================================

import os
import json
import re
import datetime

from time import time, sleep
from uuid import uuid4

from openai import OpenAI
from rdflib import Graph

import config

# Imports propios
from searchInGraph import (
    buscar_frecuentes_por_opcion,
    inferir_valor_adecuado
)

from formatHelper import (
    extraer_support_category,
    extraer_cliente,
    formatear_para_llm,
    arreglar_lista_llm,
    #merge_listas_or,
    #limpiar_lista,
    aplicar_reglas
)



In [ ]:
# ============================================
# CELDA 2 - CARGA DEL GRAFO RDF
# ============================================

graph = Graph()

graph.parse(
    config.TTL_FILE_PATH,
    format=config.TTL_FORMAT
)

print("Grafo cargado correctamente")
print(f"Número de triples: {len(graph)}")

In [ ]:
# ============================================
# CELDA 3 - CONFIGURACIÓN DEL MODELO
# ============================================

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

mi_modelo = "mistral:latest"

print(f"Modelo configurado: {mi_modelo}")

In [4]:
# ============================================
# CELDA 4 - FUNCIONES AUXILIARES
# ============================================

def open_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as infile:
        return infile.read()


def save_file(filepath, content):
    with open(filepath, 'w', encoding='utf-8') as outfile:
        outfile.write(content)


def load_json(filepath):
    with open(filepath, 'r', encoding='utf-8') as infile:
        return json.load(infile)


def save_json(filepath, payload):
    with open(filepath, 'w', encoding='utf-8') as outfile:
        json.dump(
            payload,
            outfile,
            ensure_ascii=False,
            sort_keys=True,
            indent=2
        )


def timestamp_to_datetime(unix_time):
    return datetime.datetime.fromtimestamp(
        unix_time
    ).strftime("%A, %B %d, %Y at %I:%M%p %Z")

In [5]:
# ============================================
# CELDA 5 - FUNCIÓN DE COMPLETADO LLM
# ============================================

def text_completion(prompt, engine=config.MI_MODELO):

    max_retry = 5
    retry = 0

    while True:

        try:

            response = client.chat.completions.create(
                messages=[
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ],
                model=engine
            )

            text = response.choices[0].message.content

            # Limpieza básica
            text = re.sub(r'[\r\n]+', '\n', text)
            text = re.sub(r'[\t ]+', ' ', text)

            return text

        except Exception as oops:

            retry += 1

            if retry >= max_retry:
                return f"Model error: {oops}"

            print("Error comunicando con el modelo:", oops)

            sleep(config.RETRY_DELAY_SECONDS)

In [6]:
# ============================================
# CELDA 6 - VARIABLES DE ESTADO
# ============================================

convo_length = 2

unique_conv_id = str(uuid4())

prev_conv = ""

filename = unique_conv_id + "_log.txt"

log_file_path = os.path.join(
    config.LOGS_DIR,
    filename
)

save_file(log_file_path, prev_conv)

primera = True

buscar = False

mi_opcion = None

cat_buscar = 0

graph_data = []

# Estructura:
# 0 - Int_hasCustomer
# 1 - hasSupportCategory
# 2 - hasTypeInc
# 3 - incident_hasOrigin
# 4 - hasSupportGroup
# 5 - hasTechnician

mis_datos = [None, None, None, None, None, None]

print("Sistema inicializado")
print("Estado actual:", mis_datos)

Sistema inicializado
Estado actual: [None, None, None, None, None, None]


In [7]:
# ============================================
# CELDA 7 - FUNCIÓN PRINCIPAL DEL CHAT (AUTO-OPCIÓN 1)
# ============================================
contadores = {"vecesdf": 0, "vecesnv": 0}

vecesretry = 0


def procesar_mensaje_usuario(a):

    global primera
    global mis_datos
    global graph_data
    global cat_buscar
    global mi_opcion
    global prev_conv
    mis_datos = [None, None, None, None, None, None]
    # Normalizamos la entrada del usuario para validación de salida
    a_clean = str(a).strip().lower()

    if a_clean == "q":
        print("Finalizando conversación")
        return False

    # ========================================
    # GUARDAR MENSAJE RECIBIDO
    # ========================================
    timestamp = time()
    timestring = timestamp_to_datetime(timestamp)
    message = f"USER: {timestring} - {a}"

    buscar = True

    # ========================================
    # EVALUAR RESPUESTA DEL MENÚ (SIEMPRE ASUME 1)
    # ========================================
    if graph_data:
        # Se elimina la lógica de "sí", "no" y múltiples números.
        # Siempre se toma la primera opción devuelta por GraphRAG.
        mis_datos[cat_buscar] = graph_data[0]
        #print(f"\nGraphRAG: Asignado automáticamente '{graph_data[0]}' (Opción 1) a la categoría {cat_buscar}")
        
        # Reiniciamos las opciones para buscar la siguiente categoría faltante
        graph_data = [] 

    # ========================================
    # EXTRACCIÓN DE DATOS INICIALES (0 y 1)
    # ========================================
    if mis_datos[0] is None or mis_datos[0] == 'None':
        cliente = extraer_cliente(a)
        if cliente is not None:
            mis_datos[0] = cliente

    if mis_datos[1] is None or mis_datos[1] == 'None':
        support_cat = extraer_support_category(a)
        if support_cat is not None:
            mis_datos[1] = support_cat

    # ========================================
    # VERIFICAR SI YA ESTÁ COMPLETO
    # ========================================
    if None not in mis_datos and 'None' not in mis_datos:
        print(f'\nGraphRAG: query acabada. La query es {mis_datos}')
        return False

    # ========================================
    # BUSCAR CATEGORÍA FALTANTE (SI CORRESPONDE)
    # ========================================
    if buscar:
        try:
            cat_buscar = mis_datos.index(None)
        except ValueError:
            cat_buscar = mis_datos.index('None')

        graph_data = buscar_frecuentes_por_opcion(graph, mis_datos, cat_buscar)

        if not graph_data:
            graph_data = inferir_valor_adecuado(graph, mis_datos, cat_buscar)
    
    
        if graph_data:
                graph_data = aplicar_reglas(
                    './textos/reglas_incidentes.json', # Ajusta la ruta a tu fichero
                    mis_datos, 
                    cat_buscar, 
                    graph_data, contadores
                )
                
                if graph_data == []:
                    graph_data = inferir_valor_adecuado(graph, mis_datos, cat_buscar)
                    graph_data = aplicar_reglas(
                    './textos/reglas_incidentes.json', # Ajusta la ruta a tu fichero
                    mis_datos, 
                    cat_buscar, 
                    graph_data, contadores
                )
    
    
    if graph_data:
        mi_opcion = graph_data[0]

    # ========================================
    # PREPARACIÓN DE RESPUESTA EN LOGS / CONSOLA
    # ========================================
    prev_conv = open_file(log_file_path)

    if not graph_data:
        output = "No se encontraron datos. Seguramente sea un error por parte del usuario. Pregunta si se ha introducido bien el grupo."
        print(f"\n[Asistente] {output}")
    else:
        # Solo mostramos la Opción 1 ya que es la única que tomará el sistema
        output = f"¿Cuál es el valor para {config.DICCIONARIO_PREFIJOS[cat_buscar]}?\n 1. {mi_opcion}\n(Presiona Enter para continuar, se asignará esta opción automáticamente)"
        #print(f"\n[Asistente] {output}")

    # ========================================
    # GUARDAR CONVERSACIÓN
    # ========================================
    messageBot = f"[Asistente]: {timestring} - {output}"

    save_file(
        log_file_path,
        prev_conv + "\n" + message + "\n" + messageBot
    )

    return True, mis_datos





In [8]:
## ============================================
## CELDA 8 - BUCLE INTERACTIVO
## ============================================
#
#print("====================================")
#print(" SISTEMA GraphRAG + Ollama INICIADO")
#print("====================================")
#print("Escribe 'q' para salir")
#print()
#
#primero = True
#
#while True:
#    
#    if primero:
#        entrada = input("USER: ")
#    
#    primero = False
#    
#    continuar, mis_datos = procesar_mensaje_usuario(entrada)
#    
#    
#    
#    
#    if not continuar:
#        break
#
#

In [9]:
def testear_grafo(g, prefix_uri="http://repcon.org/schema#"):
    """
    Función de diagnóstico para comprobar el estado del grafo y 
    probar las funciones de búsqueda e inferencia de incidentes.
    """
    print("\n" + "="*50)
    print(" INICIANDO TEST DEL GRAFO ".center(50, "="))
    print("="*50)

    # ========================================
    # 1. TAMAÑO DEL GRAFO
    # ========================================
    print("\n[1] Comprobando tamaño del grafo...")
    try:
        print(f"Total de tripletas cargadas: {len(g)}")
    except Exception as e:
        print(f"Error al leer la longitud del grafo: {e}")

   # ========================================
    # 2. TEST DE CONTENIDO BÁSICO (Top 5 Clientes)
    # ========================================
    print("\n[2] Obteniendo los 5 clientes más frecuentes (int_hasCustomer)...")
    query_basica = f"""
    SELECT ?cliente (COUNT(?cliente) AS ?total)
    WHERE {{
        ?incident <{prefix_uri}int_hasCustomer> ?cliente .
    }}
    GROUP BY ?cliente
    ORDER BY DESC(?total)
    LIMIT 5
    """
    
    try:
        # Convertimos a lista para evitar problemas con el generador de rdflib
        resultados = list(g.query(query_basica))
        
        if not resultados:
            print(f"  ⚠ No hay datos para el predicado '{prefix_uri}int_hasCustomer'.")
            print("  🔍 Inspeccionando los 5 predicados que MÁS se repiten en tu grafo...")
            
            query_rescate = """
            SELECT ?p (COUNT(?p) AS ?total)
            WHERE { ?s ?p ?o . }
            GROUP BY ?p
            ORDER BY DESC(?total)
            LIMIT 5
            """
            resultados_rescate = g.query(query_rescate)
            for r in resultados_rescate:
                print(f"    - {r.p} (Apariciones: {r.total})")
        else:
            for row in resultados:
                val = str(row.cliente).split("#")[-1] if "#" in str(row.cliente) else str(row.cliente).rsplit("/", 1)[-1]
                print(f"  - {val} (Apariciones: {row.total})")
                
    except Exception as e:
        print(f"  Error en consulta básica: {e}")

    # ========================================
    # 3. TEST DE TUS FUNCIONES
    # ========================================
    print("\n[3] Probando tus funciones de filtrado e inferencia...")
    
    # Simulamos el array 'mis_datos' (6 posiciones)
    # Suponemos que ya tenemos el cliente (índice 0), y queremos buscar la Categoría (índice 1)
    # Índices: [Customer, SupportCategory, TypeInc, Origin, SupportGroup, Technician]
    
    # ⚠️ IMPORTANTE: Cambia "Cliente_Prueba" por el nombre de un cliente real de tu grafo para testear
    datos_simulados = ["Cliente_Prueba", None, None, None, None, None]
    categoria_a_buscar = 1  # 1 = hasSupportCategory

    print(f"  Estado simulado (mis_datos): {datos_simulados}")
    print(f"  Índice a buscar: {categoria_a_buscar} (hasSupportCategory)")

    # 3.1 Test: buscar_frecuentes_por_opcion
    print("\n  >>> Ejecutando 'buscar_frecuentes_por_opcion'...")
    try:
        res_busqueda = buscar_frecuentes_por_opcion(g, datos_simulados, categoria_a_buscar, prefix_uri)
        print(f"  Resultado Búsqueda Exacta: {res_busqueda}")
    except Exception as e:
        print(f"  Error en buscar_frecuentes_por_opcion: {e}")

    # 3.2 Test: inferir_valor_adecuado
    print("\n  >>> Ejecutando 'inferir_valor_adecuado' (Fallback)...")
    try:
        res_inferencia = inferir_valor_adecuado(g, datos_simulados, categoria_a_buscar, prefix_uri)
        print(f"  Resultado Inferencia: {res_inferencia}")
    except Exception as e:
        print(f"  Error en inferir_valor_adecuado: {e}")

    print("\n" + "="*50)
    print(" TEST FINALIZADO ".center(50, "="))
    print("="*50 + "\n")

In [10]:
# Asumiendo que tu grafo se llama 'graph' en el entorno global:
# ⚠️ Cambia el "Cliente_Prueba" en la función por un string que sepas que sí existe en tu ontología
testear_grafo(graph)


============ INICIANDO TEST DEL GRAFO ============

[1] Comprobando tamaño del grafo...
Total de tripletas cargadas: 7425543

[2] Obteniendo los 5 clientes más frecuentes (int_hasCustomer)...
  - company__3S8A2Y7FV (Apariciones: 61072)
  - ss (Apariciones: 37008)
  - company__QCTRKWQRI (Apariciones: 28395)
  - company__CFD5UKZBE (Apariciones: 17120)
  - company__9G1G3MV0P (Apariciones: 15626)

[3] Probando tus funciones de filtrado e inferencia...
  Estado simulado (mis_datos): ['Cliente_Prueba', None, None, None, None, None]
  Índice a buscar: 1 (hasSupportCategory)

  >>> Ejecutando 'buscar_frecuentes_por_opcion'...
  Resultado Búsqueda Exacta: []

  >>> Ejecutando 'inferir_valor_adecuado' (Fallback)...
  Resultado Inferencia: []

================ TEST FINALIZADO =================



In [11]:
# ============================================
# CELDA 9 - VISUALIZAR ESTADO FINAL
# ============================================

print("====================================")
print(" ESTADO FINAL")
print("====================================")

labels = [
    "Cliente",
    "SupportCategory",
    "TypeInc",
    "Origin",
    "SupportGroup",
    "Technician"
]

for i, valor in enumerate(mis_datos):

    print(f"{labels[i]} -> {valor}")

 ESTADO FINAL
Cliente -> None
SupportCategory -> None
TypeInc -> None
Origin -> None
SupportGroup -> None
Technician -> None


In [12]:
# ============================================
# CELDA 10 - VISUALIZAR LOG
# ============================================

print("====================================")
print(" LOG DE CONVERSACIÓN")
print("====================================")

contenido_log = open_file(log_file_path)

print(contenido_log)

 LOG DE CONVERSACIÓN



In [26]:
import unittest

mis_datos = [None, None, None, None, None, None]
contadores = {"vecesdf": 0, "vecesnv": 0}

def procesar_query(texto):
    """Reutiliza la lógica determinista del notebook."""
    global mis_datos

    veces = 0
    primero = True

    while True:
        if primero:
            entrada = texto

        primero = False

        continuar = procesar_mensaje_usuario(entrada)

        if veces >= 10:
            continuar = False

        veces += 1
        if not continuar:
            break

    return mis_datos


class TestQueryProcessor(unittest.TestCase):
     def setUp(self):
        self.casos_de_prueba = [

            # 1 - 10 se activa solo DF

            (
                'Hola quiero completar una query. Tengo el supportCategory_1497617941762302663 y la empresa company_149762002231762302862',
                ['company__F7UMNAXNO', 'supportCategory_1497617941762302663', 'typeIncident__2', 'incidentOrigin__3',
                 'supportGroup_149762881762302662', 'employee__294']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149763527461762303053 y la empresa company__QWY4YPRG7',
                ['company__QWY4YPRG7', 'supportCategory_149763527461762303053', 'typeIncident__1', 'incidentOrigin__3',
                 'supportGroup_149762881762302662', 'employee__294']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149762916841762302974 y la empresa company__UQHIM9QXH',
                ['company__UQHIM9QXH', 'supportCategory_149762916841762302974', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__486']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_1497611841762302662 y la empresa company__1GR6455ID',
                ['company__1GR6455ID', 'supportCategory_1497611841762302662', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__259']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149761991762302662 y la empresa company__2ZFMBC970',
                ['company__2ZFMBC970', 'supportCategory_149761991762302662', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_1497684871762302665', 'employee__366']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149767291231762303563 y la empresa ss',
                ['ss', 'supportCategory_149767291231762303563', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976691762302662', 'employee__366']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_1497681762302662 y la empresa company_149762002231762302862',
                ['company__F7UMNAXNO', 'supportCategory_1497681762302662', 'typeIncident__1', 'incidentOrigin__3',
                 'supportGroup_149762881762302662', 'employee__294']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_1497614541762302662 y la empresa ss',
                ['ss', 'supportCategory_1497614541762302662', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976691762302662', 'employee__366']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149763671762302662 y la empresa company_149767814271762303637',
                ['company_149762002231762302862', 'supportCategory_149763671762302662', 'typeIncident__2',
                 'incidentOrigin__3', 'supportGroup_149762881762302662', 'employee__294']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149764151762302662 y la empresa company__CFD5UKZBE',
                ['company__CFD5UKZBE', 'supportCategory_149764151762302662', 'typeIncident__2', 'incidentOrigin__3',
                 'supportGroup_149762881762302662', 'employee__294']
            ),

            # 11-20 se activa solo nv
            (
                'Hola quiero completar una query. Tengo el supportCategory_1497611981762302662 y la empresa company__D52NKD9SS',
                ['company__D52NKD9SS', 'supportCategory_1497611981762302662', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__108']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_1497611981762302662 y la empresa company__17Q32M10L',
                ['company__17Q32M10L', 'supportCategory_1497611981762302662', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149761521762302662', 'employee__108']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149763671762302662 y la empresa company__17Q32M10L',
                ['company__17Q32M10L', 'supportCategory_149763671762302662', 'typeIncident__2', 'incidentOrigin__2',
                 'supportGroup_149761521762302662', 'employee__294']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_1497691561762302665 y la empresa company__17Q32M10L',
                ['company__17Q32M10L', 'supportCategory_1497691561762302665', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149761521762302662', 'employee__366']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149765491762302662 y la empresa company__077OCQVXM',
                ['company__077OCQVXM', 'supportCategory_149765491762302662', 'typeIncident__2', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__294']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149765458501762303308 y la empresa company__UQ0EGOKMC',
                ['company__UQ0EGOKMC', 'supportCategory_149765458501762303308', 'typeIncident__2', 'incidentOrigin__2',
                 'supportGroup_149761521762302662', 'employee__294']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_1497611981762302662 y la empresa company__DQJKY6U6E',
                ['company__DQJKY6U6E', 'supportCategory_1497611981762302662', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__108']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149765491762302662 y la empresa company__10QMPVMGA',
                ['company__10QMPVMGA', 'supportCategory_149765491762302662', 'typeIncident__2', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__294']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149763371762302662 y la empresa company__17Q32M10L',
                ['company__17Q32M10L', 'supportCategory_149763371762302662', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149761521762302662', 'employee__366']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149765491762302662 y la empresa company__HVY8FPJ74',
                ['company__HVY8FPJ74', 'supportCategory_149765491762302662', 'typeIncident__2', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__294']
            ),

            # 21-30 se activan ambas en el mismo momento

            (
                'Hola quiero completar una query. Tengo el supportCategory_149767060481762303533 y la empresa company_149767070781762303534',
                ['company__3S8A2Y7FV', 'supportCategory_149767060481762303533', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149765457361762303308', 'employee__429']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149767060481762303533 y la empresa company_149761171471762302766',
                ['company__3S8A2Y7FV', 'supportCategory_149767060481762303533', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149765457361762303308', 'employee__429']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149762916841762302974 y la empresa company_14976568571762302706',
                ['company__3S8A2Y7FV', 'supportCategory_149762916841762302974', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149769961762302662', 'employee__429']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_1497617231762302663 y la empresa company__3S8A2Y7FV',
                ['company__3S8A2Y7FV', 'supportCategory_1497617231762302663', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__429']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149762916841762302974 y la empresa company_149763729881762303079',
                ['company__3S8A2Y7FV', 'supportCategory_149762916841762302974', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149769961762302662', 'employee__429']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149767601762302662 y la empresa company_149766665321762303469',
                ['company__3S8A2Y7FV', 'supportCategory_149767601762302662', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149764431762302662', 'employee__429']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_14976411762302662 y la empresa company_149767814271762303637',
                ['company__3S8A2Y7FV', 'supportCategory_14976411762302662', 'typeIncident__2', 'incidentOrigin__1',
                 'supportGroup_149762761762302662', 'employee__429']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149763471762302662 y la empresa company_149764841521762303225',
                ['company__3S8A2Y7FV', 'supportCategory_149763471762302662', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149763481762302662', 'employee__429']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149763834511762303087 y la empresa company_149764242351762303146',
                ['company__3S8A2Y7FV', 'supportCategory_149763834511762303087', 'typeIncident__2', 'incidentOrigin__2',
                 'supportGroup_149762761762302662', 'employee__429']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_1497631491762302663 y la empresa company__3S8A2Y7FV',
                ['company__3S8A2Y7FV', 'supportCategory_1497631491762302663', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__429']
            ),

            # 31-32 se activa una df y una nv en momentos distintos

            (
                'Hola quiero completar una query. Tengo el supportCategory_14976110321762302666 y la empresa company__W0S6TBURD',
                ['company__W0S6TBURD', 'supportCategory_14976110321762302666', 'typeIncident__1', 'incidentOrigin__3',
                 'supportGroup_149762881762302662', 'employee__294']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_1497610291762302662 y la empresa company__UQ0EGOKMC',
                ['company__UQ0EGOKMC', 'supportCategory_1497610291762302662', 'typeIncident__2', 'incidentOrigin__3',
                 'supportGroup_149762881762302662', 'employee__294']
            ),

            # 33 - 50 no se activa ninguna regla

            (
                'Hola quiero completar una query. Tengo el supportCategory_149769471762302662 y la empresa company__IJVARZ08A',
                ['company__IJVARZ08A', 'supportCategory_149769471762302662', 'typeIncident__2', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__266']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_1497623551762302663 y la empresa company__Y1XHYEPUS',
                ['company__Y1XHYEPUS', 'supportCategory_1497623551762302663', 'typeIncident__2', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__294']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_1497622131762302663 y la empresa company_149766632941762303467',
                ['company__QCTRKWQRI', 'supportCategory_1497622131762302663', 'typeIncident__2', 'incidentOrigin__2',
                 'supportGroup_1497691762302662', 'employee__266']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149765639211762303330 y la empresa company__GBKQFD0VA',
                ['company__GBKQFD0VA', 'supportCategory_149765639211762303330', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__366']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_1497643821762302663 y la empresa company__G93IZU6VM',
                ['company__G93IZU6VM', 'supportCategory_1497643821762302663', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__366']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_14976110321762302666 y la empresa company__JAOY9VHCI',
                ['company__JAOY9VHCI', 'supportCategory_14976110321762302666', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__366']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_1497612201762302662 y la empresa company__8VD90FMG4',
                ['company__8VD90FMG4', 'supportCategory_1497612201762302662', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__366']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_1497644851762302663 y la empresa company__9G1G3MV0P',
                ['company__9G1G3MV0P', 'supportCategory_1497644851762302663', 'typeIncident__2', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__294']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_1497634611762302663 y la empresa company__FWP37ZIFM',
                ['company__FWP37ZIFM', 'supportCategory_1497634611762302663', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__366']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149762050541762302872 y la empresa company__44XJYGG3L',
                ['company__44XJYGG3L', 'supportCategory_149762050541762302872', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__366']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149762443731762302917 y la empresa company__HJLEZ1WY6',
                ['company__HJLEZ1WY6', 'supportCategory_149762443731762302917', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__366']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_1497634611762302663 y la empresa company__QRPQNRU25',
                ['company__QRPQNRU25', 'supportCategory_1497634611762302663', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__366']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149768521762302662 y la empresa company__0GQ4QH8N2',
                ['company__0GQ4QH8N2', 'supportCategory_149768521762302662', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149761521762302662', 'employee__366']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149766361762302662 y la empresa company__Y1XHYEPUS',
                ['company__Y1XHYEPUS', 'supportCategory_149766361762302662', 'typeIncident__2', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__294']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149767793181762303635 y la empresa company_149763071541762302992',
                ['company__10QMPVMGA', 'supportCategory_149767793181762303635', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149761521762302662', 'employee__403']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_1497611711762302662 y la empresa company__ZLMCEWBY6',
                ['company__ZLMCEWBY6', 'supportCategory_1497611711762302662', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__366']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_149764571762302662 y la empresa company__Y1XHYEPUS',
                ['company__Y1XHYEPUS', 'supportCategory_149764571762302662', 'typeIncident__2', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__266']
            ),
            (
                'Hola quiero completar una query. Tengo el supportCategory_1497621981762302663 y la empresa company__TLL5K8PC5',
                ['company__TLL5K8PC5', 'supportCategory_1497621981762302663', 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976631762302662', 'employee__366']
            )


        ]


    def test_reglas_df(self):
        """Casos 11-20: solo reglas df se activan."""
        print("Casos 11-20: solo reglas df se activan.")
        for i, (query, expected) in enumerate(self.casos_de_prueba[10:20], start=11):
            with self.subTest(caso=i, query=query):
                result = procesar_query(query)
                self.assertEqual(
                    result, expected,
                    msg=f"Caso {i} fallido.\nQuery: {query}\nEsperado: {expected}\nObtenido: {result}"
                )

    #def test_sin_reglas(self):
    #    """Casos 1-10: ninguna regla df ni noValid activa."""
    #    print("Casos 1-10: ninguna regla df ni noValid activa.")
    #    for i, (query, expected) in enumerate(self.casos_de_prueba[:10], start=1):
    #        with self.subTest(caso=i, query=query):
    #            result = procesar_query(query)
    #            self.assertEqual(
    #                result, expected,
    #                msg=f"Caso {i} fallido.\nQuery: {query}\nEsperado: {expected}\nObtenido: {result}"
    #            )
#

    
    def test_reglas_novalid(self):
        """Casos 21-30: solo reglas noValid se activan."""
        print("Casos 21-30: solo reglas noValid se activan.")
        for i, (query, expected) in enumerate(self.casos_de_prueba[10:20], start=21):
            with self.subTest(caso=i, query=query):
                result = procesar_query(query)
                self.assertEqual(
                    result, expected,
                    msg=f"Caso {i} fallido.\nQuery: {query}\nEsperado: {expected}\nObtenido: {result}"
                )
#
    #def test_ambas_reglas(self):
    #    """Casos 31-40: reglas df y noValid se activan sin contradiccion."""
    #    print("Casos 31-40: reglas df y noValid se activan sin contradiccion.")
    #    for i, (query, expected) in enumerate(self.casos_de_prueba[30:40], start=31):
    #        with self.subTest(caso=i, query=query):
    #            result = procesar_query(query)
    #            self.assertEqual(
    #                result, expected,
    #                msg=f"Caso {i} fallido.\nQuery: {query}\nEsperado: {expected}\nObtenido: {result}"
    #            )
#
    #def test_reglas_contradictorias(self):
    #    """Casos 41-50: df y noValid se contradicen sobre el mismo predicado."""
    #    print("Casos 41-50: df y noValid se contradicen sobre el mismo predicado.")
    #    for i, (query, expected) in enumerate(self.casos_de_prueba[40:50], start=41):
    #        with self.subTest(caso=i, query=query):
    #            result = procesar_query(query)
    #            self.assertEqual(
    #                result, expected,
    #                msg=f"Caso {i} fallido.\nQuery: {query}\nEsperado: {expected}\nObtenido: {result}"
    #            )
#

if __name__ == "__main__":
    unittest.main(argv=[""], verbosity=2, exit=False)

    
    
    print("DF veces")
    print(contadores["vecesdf"])
    print("NO VALUE veces")
    print(contadores["vecesnv"])

test_reglas_novalid (__main__.TestQueryProcessor.test_reglas_novalid)
Casos 21-30: solo reglas noValid se activan. ... 

Casos 21-30: solo reglas noValid se activan.

[Regla Aplicada - noValid] 'typeIncident__1' no es válido para 'hasTypeInc'.
[Regla Aplicada] Descartando opción principal. Saltando a la siguiente opción.

[Regla Aplicada - noValid] 'typeIncident__1' no es válido para 'hasTypeInc'.
[Regla Aplicada] Descartando opción principal. Saltando a la siguiente opción.

[Regla Aplicada - noValid] 'typeIncident__1' no es válido para 'hasTypeInc'.
[Regla Aplicada] Descartando opción principal. Saltando a la siguiente opción.

[Regla Aplicada - noValid] 'typeIncident__1' no es válido para 'hasTypeInc'.
[Regla Aplicada] Descartando opción principal. Saltando a la siguiente opción.



  test_reglas_novalid (__main__.TestQueryProcessor.test_reglas_novalid) (caso=21, query='Hola quiero completar una query. Tengo el supportCategory_1497624411762302663 y la empresa company__0CXRQYIPG')
Casos 21-30: solo reglas noValid se activan. ... FAIL

FAIL: test_reglas_novalid (__main__.TestQueryProcessor.test_reglas_novalid) (caso=21, query='Hola quiero completar una query. Tengo el supportCategory_1497624411762302663 y la empresa company__0CXRQYIPG')
Casos 21-30: solo reglas noValid se activan.
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_175/3731189183.py", line 478, in test_reglas_novalid
    self.assertEqual(
AssertionError: Lists differ: ['com[52 chars]63', None, 'incidentOrigin__1', None, None] != ['com[52 chars]63', 'typeIncident__1', 'incidentOrigin__1', '[45 chars]371']

First differing element 2:
None
'typeIncident__1'

  ['company__0CXRQYIPG',
   'supportCategory_1497624411762302663',
-


[Regla Aplicada - noValid] 'typeIncident__1' no es válido para 'hasTypeInc'.
[Regla Aplicada] Descartando opción principal. Saltando a la siguiente opción.
DF veces
0
NO VALUE veces
5
